# 1. Phân tích Khám phá Dữ liệu (EDA)
Đầu tiên, chúng ta sẽ đọc dữ liệu đã được làm sạch và vẽ **Ma trận tương quan (Heatmap)** để xem các đặc trưng (Diện tích, Số tầng...) ảnh hưởng tới Giá nhà như thế nào.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc file dữ liệu sạch đã lưu từ bước trước (Cập nhật đường dẫn nếu cần)
df = pd.read_csv('../data/processed/clean.csv')

# In ra 5 dòng đầu
display(df.head())

# Lọc các biến số để phân tích tương quan
num_cols = ['Giá_m2_trieu', 'Diện tích m2', 'Số phòng ngủ', 'Số tầng']
df_num = df[num_cols].dropna()

# Vẽ Ma trận tương quan (Heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(df_num.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Ma trận tương quan (Heatmap) giữa các biến số')
plt.show()

# 2. Xây dựng Mô hình (Modeling) & Lưu Mô hình
Chúng ta sẽ huấn luyện một mô hình **Random Forest** để dự đoán Giá nhà.
Điểm đặc biệt ở đây là ta bọc luôn cả quá trình xử lý (Scaler, OneHot) vào cùng 1 **Pipeline lớn**. Sau khi huấn luyện xong, Pipeline này sẽ được lưu ra file `.pkl` để sau này trang Web App có thể dùng trực tiếp.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1. Chuẩn bị X (Đặc trưng) và y (Mục tiêu)
features = ['Diện tích m2', 'Số phòng ngủ', 'Số tầng', 'Thieu_So_Tang', 'Quận', 'Loại hình nhà ở', 'Giấy tờ pháp lý']
X = df[features]
y = df['Giá_m2_trieu']

# 2. Xây dựng Pipeline xử lý
num_cols = ['Diện tích m2', 'Số phòng ngủ', 'Số tầng', 'Thieu_So_Tang']
cat_cols = ['Quận', 'Loại hình nhà ở', 'Giấy tờ pháp lý']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='khong_ro')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Bọc Preprocessor và Mô hình Random Forest vào chung 1 Pipeline lớn
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 3. Chia tập Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Huấn luyện (Train)
print("Đang huấn luyện mô hình Random Forest...")
model_pipeline.fit(X_train, y_train)

# 5. Đánh giá (Evaluate)
y_pred = model_pipeline.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.2f} (Càng gần 1.0 càng tốt)")

# 6. Lưu mô hình ra file để dùng cho Web App
joblib.dump(model_pipeline, 'house_price_model.pkl')
print("Đã lưu toàn bộ Pipeline mô hình vào file 'house_price_model.pkl'!")
